# Treino U-Net no Google Colab (GPU)

Pré-treino (Inria, binário) + fine-tuning (chips multiclasse com as tuas anotações).

**Antes de correr:**
1. **Runtime → Change runtime type → GPU** (T4 ou melhor).
2. **Projeto:** o notebook clona https://github.com/phmotad/roofArea.git na célula abaixo.
3. **Dados:** na célula indicada, faz upload de `chips_multiclass.zip` (pasta com `images/` e `masks/`) ou usa a pasta se já estiver no ambiente.

In [3]:
# Verificar GPU
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if out.returncode == 0:
    print('GPU:', out.stdout.strip())
else:
    print('Nenhuma GPU detectada. Menu Runtime → Change runtime type → GPU')

GPU: Tesla T4, 15360 MiB


In [4]:
# Clona o repositório roofArea (ou usa se já existir)
import os

REPO_URL = "https://github.com/phmotad/roofArea.git"
REPO_DIR = "roofArea"

if os.path.isdir(REPO_DIR) and os.path.isfile(os.path.join(REPO_DIR, "scripts", "train_unet.py")):
    print("Projeto já clonado em", REPO_DIR)
    %cd roofArea
else:
    !git clone --depth 1 {REPO_URL}
    %cd roofArea
    print("Projeto pronto.")

Cloning into 'roofArea'...
remote: Enumerating objects: 1700, done.
remote: Counting objects: 100% (1700/1700), done.
remote: Compressing objects: 100% (1633/1633), done.
remote: Total 1700 (delta 61), reused 1700 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (1700/1700), 460.63 MiB | 17.44 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Updating files: 100% (1758/1758), done.
/content/roofArea
Projeto pronto.


A célula acima clona o repositório **roofArea** do GitHub. Se já tiveres a pasta `roofArea` com o código, não é preciso clonar de novo.

In [5]:
# Instalar dependências (só as necessárias para treino)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q pillow numpy opencv-python-headless scikit-image huggingface_hub
!pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 44.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 10.0 MB/s eta 0:00:00
  Building editable for roof-api (pyproject.toml) ... done


In [6]:
# Descarregar e converter dataset Inria (pré-treino binário)
!python -m scripts.download_inria_dataset --output_dir ./dados_inria

2026-02-25 22:03:55 | INFO | A descarregar blanchon/INRIA-Aerial-Image-Labeling (apenas train) ...
2026-02-25 22:03:55 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/blanchon/INRIA-Aerial-Image-Labeling/revision/main "HTTP/1.1 200 OK"
Fetching 360 files:   0% 0/360 [00:00<?, ?it/s]2026-02-25 22:03:56 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/blanchon/INRIA-Aerial-Image-Labeling/resolve/98acad6712f12bb938bb0a8cf025bbace114c7e5/data/train/gt/austin1.tif "HTTP/1.1 302 Found"
2026-02-25 22:03:56 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/blanchon/INRIA-Aerial-Image-Labeling/resolve/98acad6712f12bb938bb0a8cf025bbace114c7e5/data/train/gt/austin11.tif "HTTP/1.1 302 Found"
2026-02-25 22:03:56 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/blanchon/INRIA-Aerial-Image-Labeling/resolve/98acad6712f12bb938bb0a8cf025bbace114c7e5/data/train/gt/austin10.tif "HTTP/1.1 302 Found"
2026-02-25 22:03:56 | INFO | HTTP Request: HEAD https://huggi

In [7]:
# Teus chips multiclasse: se já existir ./chips_multiclass (ex.: Cursor Colab), não faz nada.
# Senão, faz upload de chips_multiclass.zip (pasta com images/ e masks/).
import os
import zipfile

if os.path.isdir('chips_multiclass') and os.path.isdir('chips_multiclass/images'):
    print('chips_multiclass já presente.')
else:
    from google.colab import files
    uploaded = files.upload()  # escolhe chips_multiclass.zip
    zip_name = [k for k in uploaded if k.endswith('.zip')][0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('.')
    print('Extraído:', zip_name)

chips_multiclass já presente.


**Alternativa: Google Drive** — se tiveres a pasta no Drive, monta e copia:
```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r '/content/drive/MyDrive/caminho/para/chips_multiclass' ./chips_multiclass
```

In [1]:
# Pré-treino (binário, Inria) — GPU
!python -u -m scripts.train_unet \
  --data_dir ./dados_inria \
  --output ./models/unet_roof_pretrain.pt \
  --num_classes 1 \
  --size 512 512 \
  --epochs 30 \
  --device cuda

/usr/bin/python3: Error while finding module specification for 'scripts.train_unet' (ModuleNotFoundError: No module named 'scripts')


In [ ]:
# Fine-tuning (multiclasse, teus chips) — GPU
!python -u -m scripts.train_unet \
  --data_dir ./chips_multiclass \
  --output ./models/unet_roof_multiclass.pt \
  --num_classes 5 \
  --size 512 512 \
  --epochs 50 \
  --pretrain ./models/unet_roof_pretrain.pt \
  --device cuda

In [ ]:
# Retomar pré-treino (usa o checkpoint _last.pt da época anterior)
!python -u -m scripts.train_unet \
  --data_dir ./dados_inria \
  --output ./models/unet_roof_pretrain.pt \
  --num_classes 1 \
  --size 512 512 \
  --epochs 30 \
  --device cuda \
  --resume ./models/unet_roof_pretrain_last.pt

## Teste: inferência com o modelo treinado
Carrega o modelo, corre numa imagem de teste e mostra a máscara + área estimada.

In [ ]:
# Teste: imagem do telhado + tamanho (área estimada)
import sys
import numpy as np
import cv2
from pathlib import Path
from PIL import Image

if Path('src').is_dir():
    sys.path.insert(0, '.')
    sys.path.insert(0, 'src')

import torch
from roof_api.segmentation.unet_model import load_unet, predict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_unet('models/unet_roof_multiclass.pt', device, num_classes=5)

images_dir = Path('chips_multiclass/images')
test_images = sorted(images_dir.glob('*.png'))[:3]
if not test_images:
    test_images = list(Path('.').glob('**/images/*.png'))[:3]

METROS_POR_PIXEL = 0.10

for path in test_images:
    rgb = np.array(Image.open(path).convert('RGB'), dtype=np.uint8)
    prob = predict(model, rgb, device)
    mask = (prob > 0.6).astype(np.uint8)
    area_px = int(mask.sum())
    area_m2 = area_px * (METROS_POR_PIXEL ** 2)
    overlay = rgb.copy().astype(np.float32)
    overlay[mask > 0] = overlay[mask > 0] * 0.5 + np.array([255, 80, 80], dtype=np.float32) * 0.5
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)
    print(f'{path.name}: área estimada {area_m2:.1f} m² ({area_px} px, {METROS_POR_PIXEL} m/px)')
    from IPython.display import display
    display(Image.fromarray(overlay))

In [ ]:
# Descarregar o modelo treinado para o teu PC
from google.colab import files
files.download('models/unet_roof_multiclass.pt')

Opcional: guardar no Google Drive em vez de download:
```python
!cp models/unet_roof_multiclass.pt '/content/drive/MyDrive/'
```